# 3장 2강: SDK 개발 환경 구성과 연결 테스트
## 3. Supabase Client 초기화

In [1]:
import os
from supabase import create_client

url = os.getenv('SUPABASE_URL')
key = os.getenv('SUPABASE_PUBLISHABLE_KEY')

# Client 초기화
supabase = create_client(url, key)

print(supabase)


## 4. 연결 테스트와 응답·오류 객체 확인

# 3장 3강: SDK 기반 데이터 생성 구현

## 1. SDK insert 구조

In [3]:
# 사용자 추가
response = supabase.table("users").insert({"email": "user03@test.org"}).execute()

In [5]:
response.data

[{'id': '9c36cf78-204d-47f5-8f0f-dbfc27ecbae4',
  'email': 'user03@test.org',
  'created_at': '2026-09-09T01:54:52.773663'}]

In [21]:
# documents에 추가
response = supabase.table("documents").insert({
    "title": "두 번째 문서",
    "content": "본문 내용",
    "user_id": "9c36cf78-204d-47f5-8f0f-dbfc27ecbae4"
}).execute()

response.data

[{'id': '08537127-c06e-4391-8a01-980cae0780f0',
  'title': '두 번째 문서',
  'content': '본문 내용',
  'user_id': '9c36cf78-204d-47f5-8f0f-dbfc27ecbae4',
  'status': 'draft',
  'created_at': '2026-09-09T02:37:44.1259+00:00'}]

In [7]:
# 회원 목록 조회
response = supabase.table("users").select("*").execute()

response.data

[{'id': 'b65bbc63-f843-479f-8cf8-1ea74ed7cd8d',
  'email': 'user01@test.org',
  'created_at': '2026-09-08T03:00:51.6603'},
 {'id': 'b504df8c-21a5-4bbd-bc14-0cc795ba40f5',
  'email': 'user02@test.org',
  'created_at': '2026-09-09T00:42:40.130561'},
 {'id': '9c36cf78-204d-47f5-8f0f-dbfc27ecbae4',
  'email': 'user03@test.org',
  'created_at': '2026-09-09T01:54:52.773663'}]

In [ ]:
response = (
    supabase.table("users")
        .select("*")
        .ilike("email", "user04%")
        .single() # 레코드 1개, 무조건 1개가 나와야 한다, 아니면 예외발생, 반환값은 딕셔너리
        .execute()
)

print(response.data)

# id, email, created_at = response.data[0]

# print(f"id:{id}, email:{email}, created_at:{created_at}")

APIError: {'message': 'Cannot coerce the result to a single JSON object', 'code': 'PGRST116', 'hint': None, 'details': 'The result contains 0 rows'}

In [19]:
# .maybe_single() 0~1, 0개이면 None 
response = (
    supabase.table("users")
        .select("*")
        .ilike("email", "user%")
        .limit(1)
        .offset(1) # 1 인덱스에서 1개를 조회
        .maybe_single()
        .execute()
)

print(response)

data={'id': 'b504df8c-21a5-4bbd-bc14-0cc795ba40f5', 'email': 'user02@test.org', 'created_at': '2026-09-09T00:42:40.130561'} count=None


In [20]:
response = (
    supabase.table("users")
        .select("*")
        .order("email", desc=True)
        .execute()
)

response.data

[{'id': '9c36cf78-204d-47f5-8f0f-dbfc27ecbae4',
  'email': 'user03@test.org',
  'created_at': '2026-09-09T01:54:52.773663'},
 {'id': 'b504df8c-21a5-4bbd-bc14-0cc795ba40f5',
  'email': 'user02@test.org',
  'created_at': '2026-09-09T00:42:40.130561'},
 {'id': 'b65bbc63-f843-479f-8cf8-1ea74ed7cd8d',
  'email': 'user01@test.org',
  'created_at': '2026-09-08T03:00:51.6603'}]

In [22]:
# LEFT JOIN

response = (
    supabase.table("documents")
        .select("id, title, content, users(id, email)")
        .execute()
)

response.data

[{'id': '63135482-c74f-44a9-965f-8dd80920d68c',
  'title': '첫 번째 문서',
  'content': '본문 내용',
  'users': {'id': '9c36cf78-204d-47f5-8f0f-dbfc27ecbae4',
   'email': 'user03@test.org'}},
 {'id': '08537127-c06e-4391-8a01-980cae0780f0',
  'title': '두 번째 문서',
  'content': '본문 내용',
  'users': {'id': '9c36cf78-204d-47f5-8f0f-dbfc27ecbae4',
   'email': 'user03@test.org'}}]

In [24]:
# INNER JOIN(!inner)
response = (
    supabase.table("documents")
        .select("id, title, content, users!inner(id, email)")
        .ilike("users.email", "user%")
        .execute()
)

response.data

[{'id': '63135482-c74f-44a9-965f-8dd80920d68c',
  'title': '첫 번째 문서',
  'content': '본문 내용',
  'users': {'id': '9c36cf78-204d-47f5-8f0f-dbfc27ecbae4',
   'email': 'user03@test.org'}},
 {'id': '08537127-c06e-4391-8a01-980cae0780f0',
  'title': '두 번째 문서',
  'content': '본문 내용',
  'users': {'id': '9c36cf78-204d-47f5-8f0f-dbfc27ecbae4',
   'email': 'user03@test.org'}}]

## 2. 입력값과 DB 컬럼 매핑

## 3. 생성 결과와 오류 응답 처리

In [ ]:
create_document(None, "본문 내용입니다.", 1)

# 3장 4강: SDK 기반 데이터 조회 구현

## 1. SDK select 구조

## 2. 조건 필터링: eq, order, limit

## 3. 사용자별 조회와 결과 리스트 처리

# 3장 5강: SDK 기반 데이터 수정·삭제 구현

## 1. SDK update 구조

In [27]:
reponse = (
    supabase.table("documents")
        .select("*")
        .execute()
)

print(response.data)

[{'id': '63135482-c74f-44a9-965f-8dd80920d68c', 'title': '첫 번째 문서(수정)', 'content': '본문 내용', 'user_id': '9c36cf78-204d-47f5-8f0f-dbfc27ecbae4', 'status': 'draft', 'created_at': '2026-09-09T01:59:33.87612+00:00'}]


In [26]:
response = (
    supabase.table("documents")
        .update({
            "title": "첫 번째 문서(수정)"
        })
        .eq("id", "63135482-c74f-44a9-965f-8dd80920d68c")
        .execute()
)

response.data

[{'id': '63135482-c74f-44a9-965f-8dd80920d68c',
  'title': '첫 번째 문서(수정)',
  'content': '본문 내용',
  'user_id': '9c36cf78-204d-47f5-8f0f-dbfc27ecbae4',
  'status': 'draft',
  'created_at': '2026-09-09T01:59:33.87612+00:00'}]

## 2. SDK delete 구조

In [30]:
response = (
    supabase.table("users")
        .select("*")
        .execute()
)

response.data

[{'id': 'b504df8c-21a5-4bbd-bc14-0cc795ba40f5',
  'email': 'user02@test.org',
  'created_at': '2026-09-09T00:42:40.130561'},
 {'id': '9c36cf78-204d-47f5-8f0f-dbfc27ecbae4',
  'email': 'user03@test.org',
  'created_at': '2026-09-09T01:54:52.773663'}]

In [29]:
response = (
    supabase.table("users")
        .delete()
        .eq("id", "b65bbc63-f843-479f-8cf8-1ea74ed7cd8d")
        .execute()
)

response.data

[{'id': 'b65bbc63-f843-479f-8cf8-1ea74ed7cd8d',
  'email': 'user01@test.org',
  'created_at': '2026-09-08T03:00:51.6603'}]

## 3. 수정·삭제 후 검증과 오류 처리

In [ ]:
# 검증: 상태가 잘 바뀌었는지 다시 조회


# 3장 6강: Supabase Authentication 기본 흐름

## 2. 이메일 기반 회원가입과 로그인

In [ ]:
key = os.getenv("SUPABASE_ANON_KEY")
supabase = create_client(url, key)

"""
supabase.auth : 인증(로그인) / 인가(접근 제한) 관련 메서드 가지고 있는 객체 

테이블명: auth.users
"""

# 회원가입 
response = supabase.auth.sign_up({
    "email": "user01@test.org",
    "password": "password1234",
    "phone": "01010001000"
})

response

AuthResponse(user=User(id='320a559d-8466-4704-a87d-0bd2914605cd', app_metadata={'provider': 'email', 'providers': ['email']}, user_metadata={'email': 'user01@test.org', 'email_verified': True, 'phone_verified': False, 'sub': '320a559d-8466-4704-a87d-0bd2914605cd'}, aud='authenticated', confirmation_sent_at=None, recovery_sent_at=None, email_change_sent_at=None, new_email=None, new_phone=None, invited_at=None, action_link=None, email='user01@test.org', phone='', created_at=datetime.datetime(2026, 9, 9, 3, 39, 57, 397320, tzinfo=TzInfo(0)), confirmed_at=None, email_confirmed_at=datetime.datetime(2026, 9, 9, 3, 39, 57, 421764, tzinfo=TzInfo(0)), phone_confirmed_at=None, last_sign_in_at=datetime.datetime(2026, 9, 9, 3, 39, 57, 428787, tzinfo=TzInfo(0)), role='authenticated', updated_at=datetime.datetime(2026, 9, 9, 3, 39, 57, 451391, tzinfo=TzInfo(0)), identities=[UserIdentity(id='320a559d-8466-4704-a87d-0bd2914605cd', identity_id='90091efd-c331-45fa-840d-48a2755abcc2', user_id='320a559d-8

In [34]:
# 로그인
response = supabase.auth.sign_in_with_password({
    "email": "user01@test.org",
    "password": "password1234"
})

response.user

User(id='320a559d-8466-4704-a87d-0bd2914605cd', app_metadata={'provider': 'email', 'providers': ['email']}, user_metadata={'email': 'user01@test.org', 'email_verified': True, 'phone_verified': False, 'sub': '320a559d-8466-4704-a87d-0bd2914605cd'}, aud='authenticated', confirmation_sent_at=None, recovery_sent_at=None, email_change_sent_at=None, new_email=None, new_phone=None, invited_at=None, action_link=None, email='user01@test.org', phone='', created_at=datetime.datetime(2026, 9, 9, 3, 39, 57, 397320, tzinfo=TzInfo(0)), confirmed_at=datetime.datetime(2026, 9, 9, 3, 39, 57, 421764, tzinfo=TzInfo(0)), email_confirmed_at=datetime.datetime(2026, 9, 9, 3, 39, 57, 421764, tzinfo=TzInfo(0)), phone_confirmed_at=None, last_sign_in_at=datetime.datetime(2026, 9, 9, 3, 48, 9, 991391, tzinfo=TzInfo(0)), role='authenticated', updated_at=datetime.datetime(2026, 9, 9, 3, 48, 10, 13652, tzinfo=TzInfo(0)), identities=[UserIdentity(id='320a559d-8466-4704-a87d-0bd2914605cd', identity_id='90091efd-c331-45

## 3. 세션, 로그아웃, user 객체

In [37]:
# 현재 로그인된 사용자 확인
user = supabase.auth.get_user() # 현재 로그인한 사용자 정보, 있으면 로그인 상태, None이면 미 인증 상태
if user: # 로그인 상태
    print("로그인 상태:", user)
else:
    print("미 로그인 상태")


미 로그인 상태


In [36]:
# 로그아웃
supabase.auth.sign_out()

## 4. 사용자 ID와 테이블 데이터 연결

# 3장 7강: 사용자별 데이터 접근과 권한 관리

## 2. user_id 기반 필터링

In [ ]:


# 본인 문서만 조회


# 3장 8강: SDK 기반 미니 과제 - 사용자별 문서 CRUD

## 2. Auth와 user_id 연결

In [ ]:

# 생성: 로그인 사용자의 문서


# 조회: 본인 문서만


## 3. 응답·오류 처리와 코드 구조 정리